# E-Commerce RAG QnA ChatBot using LangChain and FAISS vector store

## 1. Load Modules

In [1]:
import pandas as pd

import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq

from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_community.chat_message_histories import SQLChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_core.messages import AIMessage
from langchain_core.messages import HumanMessage

from IPython.display import display, Markdown

## 2. Global Settings and Configurations

In [2]:
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [3]:
ollama_embed_model = 'nomic-embed-text:latest'
embed_ollama = OllamaEmbeddings(model=ollama_embed_model)

embed_ollama

OllamaEmbeddings(model='nomic-embed-text:latest', base_url=None, client_kwargs={})

In [4]:
llm_model = "llama-3.1-8b-instant"
llm = ChatGroq(model=llm_model, temperature=0.2)

llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002121E572CE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002121E571720>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

## 3. App Input Data

In [5]:
df = pd.read_csv(r'./data/product_review.csv')

print(f'No. of Records: {df.shape[0]}')
print(f'No. of Features: {df.shape[1]} \n')

df.head()

No. of Records: 450
No. of Features: 5 



,id,title,rating,one_liner,review
0,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,1-more flexible2-bass is very high3-sound clar...
1,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Super sound and good looking I like that prize
2,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,Very much satisfied with the device at this pr...
3,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,"Nice headphone, bass was very good and sound i..."
4,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Sound quality super battery backup super quali...


## 4. App Input Data Pre-Processing

In [6]:
# Preparing Documents from the CSV file features

def docs_prep(data): 
    
    product_list = []
    for index, row in data.iterrows():
        obj = {
                'title': row['title'],
                'rating': row['rating'],
                'one_liner': row['one_liner'],
                'review': row['review']                
            }
        product_list.append(obj)
    ############################################################3
    docs = []
    for entry in product_list:
        
        te_xt = entry['review']
        meta_data = {'title': entry['title'], 'rating': entry['rating'], 'one_liner': entry['one_liner']}
        
        doc = Document(page_content=te_xt, metadata=meta_data)
        docs.append(doc)
        
    return docs

In [7]:
docs = docs_prep(df)

print(f'No. of Documents: {len(docs)}')

No. of Documents: 450


## 5. FAISS Vector Database

### 5.1 Create

In [8]:
# faiss_vector_store = FAISS.from_documents(documents=docs, embedding=embed_ollama)
# faiss_vector_store

### 5.2 Save Local

In [9]:
STORE_PATH = r'faiss_vector_store'

In [10]:
#faiss_vector_store.save_local(STORE_PATH)

### 5.3 Load Local

In [11]:
load_faiss_vector_store = FAISS.load_local(folder_path=STORE_PATH, embeddings=embed_ollama, 
                                    allow_dangerous_deserialization=True)

## 6. Retriever Response

### 6.1 Prompt Template

In [12]:
sys_template = '''
You are a polite, helpful and expert AI assistant in the domain of E-Commerce and Customer Management, and your name is "Karen".

Your job is to answer the user queries related to ecommerce platform 'Flipkart' only. You can use emojis to answer!
Introduce yourself, only in the start of conversation or when asked!
You should never talk about yourself such as training, documents, sources, architecture, last updates, or who you are in depth!
You should sound confident and bold in your answers, without any hesitation!
For any disclaimers, show them at the start of the answers only!
Keep your answer short and precise based on facts as much as possible regarding the query – do not hallucinate features!

Provide a very short and suitable excuse for all non-related queries without any further suggestions, recommendations, or guidance!
'''

### 6.2 Chain to Retrieve Relevant Documents

In [13]:
retriever = load_faiss_vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


retrieved_docs = retriever | format_docs

retrieved_docs

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002121E7D7FD0>, search_kwargs={'k': 3})
| RunnableLambda(format_docs)

## 7. ChatBot Functionality

### 7.1 Chat History Config

In [14]:
chat_message_history = SQLChatMessageHistory(session_id="test_session_id", connection="sqlite:///ecom_chats.db")

chat_message_history

### 7.2 Final Prompt for LLM

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", sys_template),
        MessagesPlaceholder(variable_name="history"),
        ("human","{question}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

### 7.3 LCEL Chain

In [16]:
chain = prompt | llm | StrOutputParser()

chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

### 7.4 Runnable Message History Chain and Config

In [17]:
chain_with_history = RunnableWithMessageHistory(chain, 
                lambda session_id: SQLChatMessageHistory(session_id=session_id, connection="sqlite:///ecom_chats.db"),
                input_messages_key="question", 
                history_messages_key="history")

config = {"configurable": {"session_id": "test_session_id"}}

chain_with_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function <lambda> at 0x000002121E90B6D0>, input_messages_key='question', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

## 8. LLM Response

### 8.1 RAG QnA

In [18]:
question = 'suggest some good earphones'

response = chain_with_history.invoke({"context":retrieved_docs, "question": question}, config=config)

display(Markdown(response))

🎧 **Disclaimer: As a Flipkart expert, I can only provide information on products available on Flipkart.**

You can check out the following earphones on Flipkart:

1. JBL C100SI In-Ear Earphones
2. Boat Rockerz 255 Pro Earphones
3. Sony MDR-XB90EX Earphones
4. Sennheiser IE 400 Pro Earphones
5. JBL Tune 510BT Earphones

Please visit Flipkart to check the latest prices and reviews! 🛍️

### 8.2 Chat History in DB

In [19]:
for msg in chat_message_history.messages:
    if isinstance(msg, HumanMessage):
        display(Markdown(f'***USER***: {msg.content}'))
    else:
        display(Markdown(f'***AI***: {msg.content}'))
        print('\n')

***USER***: suggest some good earphones

***AI***: 🎧 **Disclaimer: As a Flipkart expert, I can only provide information on products available on Flipkart.**

You can check out the following earphones on Flipkart:

1. JBL C100SI In-Ear Earphones
2. Boat Rockerz 255 Pro Earphones
3. Sony MDR-XB90EX Earphones
4. Sennheiser IE 400 Pro Earphones
5. JBL Tune 510BT Earphones

Please visit Flipkart to check the latest prices and reviews! 🛍️

### 8.3 Clear Message History

### 8.4 Streaming Response

## 9. Questions by User to the Chat Bot